# CAD primitive learning

In this notebook I will explore the second step of my proposed idea, where PN++ and Deepcad are trained together to learn extrusion primitives.

**IMPORTANT** 
For this I again turned of the random sampling in the dataset, in order to get all points.

## Dataset creation

In [300]:
import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append("..")
sys.path.append("../code")

from dataset import PCExtrusionSegmentationDataset, BaseDataset, PointCloudEmbeddingSequenceDataset

import os
import shutil
import h5py
import numpy as np
import sys
import pandas as pd
import json
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.append("..")
sys.path.append("../code")

import open3d as o3d
import torch
import random

In [2]:
train_dataset = PCExtrusionSegmentationDataset("../data", 'train', use_normals=False, verbose=False)
val_dataset = PCExtrusionSegmentationDataset("../data", 'validation', use_normals=False, verbose=False)
test_dataset = PCExtrusionSegmentationDataset("../data", 'test', use_normals=False, verbose=False)
datasets = [train_dataset, val_dataset, test_dataset]

In [3]:
def save_pc(pc, path):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pc)
    o3d.io.write_point_cloud(path, pcd)

In [4]:
def save_h5(extr_id, sequence, path):
    with h5py.File(path, "w") as f:
        f.create_dataset("extrusion_id", data=extr_id)
        f.create_dataset("sequence", data=sequence)

In [8]:
DATA_DIR = "data_exp"
error_dict = {}

for dataset in datasets:
    length = len(dataset)
    
    for i in range(length):
        
        try:
            data = dataset[i]
            id = data['id']
            print(id)
            pc = data['pc']
            label = data['label']
    
            pcs = split_pc_by_labels(pc, label)
    
            sequences = {}
            h5_path = os.path.join("..", "data", "pc_from_vec_labels", id[:4], id + ".h5") # Replace ../data with DATA_DIR
            with h5py.File(h5_path, 'r') as f:
                for class_id, sequence in f['sequences'].items():
                    sequence = sequence[:]
                    seq_length = np.where(sequence[:, 0] == 3)[0][0] + 1 # only save up to including the first EOS command
                    sequence = sequence[:seq_length, :]
                    sequences[class_id] = sequence
            
            for i, pc in enumerate(pcs):
                pc_path = os.path.join(os.path.abspath(DATA_DIR), "pc_extrusion", id[:4], id, id + "_" + str(i) + ".ply")
                os.makedirs(os.path.dirname(pc_path), exist_ok=True)
                save_pc(pc, pc_path)
                
                h5_path = os.path.join(os.path.abspath(DATA_DIR), "pc_extrusion_labels", id[:4], id, id + "_" + str(i) + ".h5")
                os.makedirs(os.path.dirname(h5_path), exist_ok=True)
                sequence = sequences[str(i)]
                save_h5(i, sequence, h5_path)
                
        except Exception as e:
            error_dict[i] = e


        if i == 1:
            break
    break

00675619
00981499
00435622
00020470
00796013
00034076


In [213]:
DATA_DIR = "../data"
def sanity_check(id):
    gt_pc_path = os.path.join("..", "data", "pc_from_vec", id[:4], id + ".ply")
    gt_lable_path = os.path.join("..", "data", "pc_from_vec_labels", id[:4], id + ".h5")

    gt_pc = o3d.io.read_point_cloud(gt_pc_path)
    gt_pc = np.asarray(gt_pc.points)

    with h5py.File(gt_lable_path, "r") as f:
        print(f.keys())
        gt_label = f['labels'][:]
    
    visualize_labeled_pc(gt_pc, gt_label)
    

    
    
    pc_dir = os.path.join(DATA_DIR, "pc_extrusion", id[:4], id)
    pc_paths = sorted([os.path.join(pc_dir, fname) for fname in os.listdir(pc_dir)])
    h5_dir = os.path.join(os.path.abspath(DATA_DIR), "pc_extrusion_labels", id[:4], id)
    h5_paths = sorted([os.path.join(h5_dir, fname) for fname in os.listdir(h5_dir)])

    for pc_file, h5_file in zip(pc_paths, h5_paths):
        point_cloud = o3d.io.read_point_cloud(pc_file)
        point_cloud = np.asarray(point_cloud.points)
        
        with h5py.File(h5_file, "r") as f:
            extr_id = f['extrusion_id'][()]
            sequence = f['sequence'][:]
        print(extr_id)
        print(sequence)
        print()
        
        visualize_pc(point_cloud)

In [361]:
sanity_check("00876358")

<KeysViewHDF5 ['labels', 'sequences']>
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
0
[[  4  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 223 128  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 223 171  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 128 171  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 128 128  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  5  -1  -1  -1  -1  -1 192  64 192  66 128 100 124 169 128   0   0]
 [  3  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]]

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
1
[[  4  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 223 128  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
 [  0 223 15

In [394]:
from glob import glob
N_POINTS = 2048
class PCExtrusionSequenceDataset():
    def __init__(self, root, split, verbose=False):
        self.pc_path = os.path.join(root, "pc_extrusion")
        self.split_path = os.path.join(root, "train_val_test_split.json")
        self.verbose = verbose
        self.split = split

        # Scrutinize if the id's mentioned in the split actually exist as directories
        valid_dirs = []
        pc_all = self.read_split()
        for dir in pc_all:
            if os.path.isdir(dir):
                valid_dirs.append(dir)
        
        self.pc = []
        self.labels = []
        
        for dir in valid_dirs:
            pc_paths = glob(os.path.join(dir, "*.ply"))
            for ply_path in pc_paths:
                h5_path = ply_path.replace("pc_extrusion", "pc_extrusion_labels").replace(".ply", ".h5")
        
                if not os.path.exists(h5_path):
                    continue
        
                try:
                    with h5py.File(h5_path, "r") as f:
                        if "sequence" in f and len(f["sequence"]) > 0:
                            self.pc.append(ply_path)
                            self.labels.append(h5_path)
                except Exception as e:
                    if self.verbose:
                        print(f"Skipped corrupted or unreadable file: {h5_path} ({e})")

            
            

    def __getitem__(self, idx):
        point_cloud = o3d.io.read_point_cloud(self.pc[idx])
        point_cloud = np.asarray(point_cloud.points)
        point_cloud = self.adjust_pointcloud_to_fixed_size(point_cloud, N_POINTS)
        point_cloud = torch.tensor(point_cloud, dtype = torch.float32) # [N, C]

        with h5py.File(self.labels[idx], "r") as f:
            extr_id = f['extrusion_id'][()]
            sequence = f['sequence'][:]

        extr_id = torch.tensor(extr_id, dtype=torch.long)
        sequence = torch.tensor(sequence, dtype=torch.long)
        
        return {'pc': point_cloud,
               'extrusion_id': extr_id,
               'sequence': sequence}

    def read_split(self):
        with open(self.split_path, "r") as fp:
            all_data = json.load(fp)
        if self.verbose:
            print(f"Number of samples that should be in the {self.split} set: {len(all_data[self.split])}", flush=True)
        pc_set = [os.path.join(self.pc_path, f"{idx}") for idx in all_data[self.split]]
        return pc_set

    def filter_pc(self, pc_set):
        pc_set_filtered = [entry for entry in pc_set if entry in self.all_pc_files]
        corrupt_files = [i for i, entry in enumerate(pc_set) if entry not in self.all_pc_files]
        if self.verbose:
            print(f"Files on disk: {len(pc_set_filtered)} --> There are {len(corrupt_files)} missing point cloud files in the {self.split} set.\n", flush=True)
        return pc_set_filtered, corrupt_files

    def __len__(self):
        assert len(self.pc) == len(self.labels), "Number of point clouds and labels doesn't match"
        return len(self.pc)

    def get_pc_path(self, idx):
        return self.pc[idx] 

    def get_label_path(self, idx):
        return self.labels[idx]

    def adjust_pointcloud_to_fixed_size(self, points, target_n):
        """
        Adjust a point cloud and its labels to a fixed number of points.
    
        - If len(points) > target_n: randomly downsample
        - If len(points) < target_n: randomly upsample with replacement
    
        :param points: (N, 3) np.ndarray
        :param labels: (N,) np.ndarray
        :param target_n: int, desired number of points
        :return: (target_n, 3) points, (target_n,) labels
        """
        n = points.shape[0]
    
        if n == target_n:
            return points, labels
        elif n > target_n:
            idx = np.random.choice(n, target_n, replace=False)
        else:
            idx_extra = np.random.choice(n, target_n - n, replace=True)
            idx = np.concatenate([np.arange(n), idx_extra])
    
        return points[idx]

In [412]:
c = PCExtrusionSequenceDataset("../data", "train", verbose=True)

Number of samples that should be in the train set: 161240


In [444]:
index = 32032
pc = c[index]['pc']
for a in c[index]['sequence'].numpy():
    print(a)

[ 4 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
[  2 176 128  -1  -1  47  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[ 4 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
[  2 141 128  -1  -1   8  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[ 4 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
[  2 176 128  -1  -1  17  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[ 4 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
[  2 176  93  -1  -1   8  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[ 4 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
[  2 176 163  -1  -1   8  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[ 4 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]
[  2 210 128  -1  -1   8  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1  -1]
[  5  -1  -1  -1  -1  -1 128 128 128  50 128 128 156 153 128   0   0]
[ 3 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1 -1]


In [445]:
visualize_pc(pc.numpy())

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [ ]:
c.get_label_path(2), c.get_pc_path(2)

In [159]:
a = PCExtrusionSequenceDataset("../data", "train", verbose=True)
b = PCExtrusionSequenceDataset("../data", "validation", verbose=True)
c = PCExtrusionSequenceDataset("../data", "test", verbose=True)

Number of samples that should be in the train set: 161240
Number of samples that should be in the validation set: 8946
Number of samples that should be in the test set: 8052


In [166]:
arsch = 143211
a[arsch], a.get_label_path(arsch), a.get_pc_path(arsch)

({'pc': tensor([[ 0.0497, -0.1402,  0.1420],
          [ 0.0163, -0.0294,  0.1492],
          [-0.0665, -0.0893, -0.1337],
          ...,
          [ 0.1043, -0.1024,  0.1093],
          [-0.0473, -0.0664, -0.1418],
          [ 0.0429, -0.1269,  0.1442]]),
  'extrusion_id': tensor(2),
  'sequence': tensor([[  4,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,
            -1,  -1,  -1],
          [  2, 176, 128,  -1,  -1,  48,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,
            -1,  -1,  -1],
          [  5,  -1,  -1,  -1,  -1,  -1, 192,  64, 192, 109, 105, 128,  38, 105,
           128,   2,   0],
          [  3,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,  -1,
            -1,  -1,  -1]])},
 '../data/pc_extrusion_labels/0058/00581069/00581069_2.h5',
 '../data/pc_extrusion/0058/00581069/00581069_2.ply')

In [167]:
visualize_pc(a[arsch]['pc'].numpy())

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [122]:
datasets = [a,b,c]
error = []
for dataset in datasets:
    length = len(dataset)
    for i in tqdm(range(length)):
        data = dataset[i]
        if not data:
            error.append(dataset.get_pc_path(i))        

100%|███████████████████████████████████| 16596/16596 [00:13<00:00, 1238.08it/s]


In [123]:
error

[]

In [87]:

for i in tqdm(range(len(c))):
    pc_path = os.path.dirname(c.get_pc_path(i)).replace("pc_extrusion", "pc_from_vec_labels")

    if not pc_path in files_on_disk:
        missing.append(c.get_pc_path(i))




100%|███████████████████████████████████| 16596/16596 [00:12<00:00, 1319.85it/s]


In [88]:
missing

[]

In [75]:
files_on_disk = []
for path in all_original_seq_files:
    files_on_disk.append(os.path.splitext(path)[0])


In [55]:
file_pattern = os.path.join("**", "*") + ".h5"

In [57]:
all_original_seq_files = glob(f"{os.path.join('../data/cad_vec', file_pattern)}", recursive=True)

In [59]:
all_original_seq_files[5]

'../data/cad_vec/0095/00959814.h5'

In [62]:
from tqdm import tqdm
counter = 0

for path in tqdm(all_original_seq_files):
    with h5py.File(path) as f:
        seq = f['vec'][:]
    for command in seq:
        if command[0] == 5:
            counter += 1

100%|█████████████████████████████████| 179133/179133 [00:58<00:00, 3037.41it/s]


In [63]:
counter

366656

In [94]:
from tqdm import tqdm

all_original_seq_files = glob(f"{os.path.join('../data/pc_from_vec_labels', file_pattern)}", recursive=True)
counter = 0

for path in tqdm(all_original_seq_files):
    with h5py.File(path) as f:
        for k, v in sequences.items():
            counter += 1
counter

100%|█████████████████████████████████| 177776/177776 [00:42<00:00, 4161.26it/s]


355552

In [ ]:
NEXT: CHECK IF NUMBER OF EXTRUSION IN ENTIRE DATASET MATCHES NUMBER OF PC
ADD targets
INclude up/downsampling of points

In [138]:
import glob, os, h5py
from tqdm import tqdm

all_seq_files = glob.glob(
    os.path.join("../data/pc_from_vec_labels", "**", "*.h5"),
    recursive=True
)

counter = 0
for path in tqdm(all_seq_files):
    with h5py.File(path, "r") as f:
        # iterate exactly what's in the file:
        for _ in f['sequences'].keys():
            counter += 1

print("Total sequences:", counter)


100%|█████████████████████████████████| 177776/177776 [00:51<00:00, 3436.36it/s]

Total sequences: 363240


In [145]:
with h5py.File("../data/pc_from_vec_labels/0000/00000007.h5") as f:
    print(f.keys())

<KeysViewHDF5 ['labels', 'sequences']>


In [196]:
import glob, os, h5py
from tqdm import tqdm

all_seq_files = glob.glob(
    os.path.join("../data/pc_from_vec_labels", "**", "*.h5"),
    recursive=True
)
all_paths = []
counter = 0
for path in tqdm(all_seq_files):
    with h5py.File(path, "r") as f:
        # iterate exactly what's in the file:
        for key in f['sequences'].keys():
            teil = os.path.splitext(path)
            id = path.split("/")[-1].split(".")[0]
            new_path = teil[0] + "/" + id + "_" + key + teil[1]
            all_paths.append(new_path)
            counter += 1

print("Total sequences:", counter)


100%|█████████████████████████████████| 177776/177776 [00:46<00:00, 3813.82it/s]

Total sequences: 363240


In [197]:
all_paths[0]

'../data/pc_from_vec_labels/0095/00956959/00956959_0.h5'

In [198]:
a.get_label_path(3)

'../data/pc_extrusion_labels/0067/00675619/00675619_3.h5'

In [200]:
datasets = [a,b,c]
error = []
all_dataset_paths = []
for dataset in datasets:
    length = len(dataset)
    for i in tqdm(range(length)):
        dataset_path = dataset.get_label_path(i)
        new_dataset_path = dataset_path.replace("pc_extrusion_labels", "pc_from_vec_labels")
        all_dataset_paths.append(new_dataset_path)


100%|████████████████████████████████| 16596/16596 [00:00<00:00, 2376614.74it/s]


In [201]:
len(all_dataset_paths), len(all_paths)

(362225, 363240)

In [204]:
error = []
for path_on_disk in tqdm(all_paths):
    if path_on_disk not in all_dataset_paths:
        error.append(path_on_disk)

100%|██████████████████████████████████| 363240/363240 [08:50<00:00, 684.24it/s]


In [205]:
len(error)

1015

In [228]:
i_list = []
for i in range(len(a)):
    lol = a.get_label_path(i)
    lol = os.path.dirname(lol)
    if lol == '../data/pc_extrusion_labels/0095/00956159':
        print("JO", i)
        i_list.append(i)

JO 173442
JO 173443
JO 173444
JO 173445


In [244]:
for k,v in a[173442].items():
    print(v.shape)

torch.Size([513, 3])
torch.Size([])
torch.Size([7, 17])


In [272]:
a.get_label_path(173442), a.get_label_path(173443), a.get_label_path(173444), a.get_label_path(173445)

('../data/pc_extrusion_labels/0095/00956159/00956159_3.h5',
 '../data/pc_extrusion_labels/0095/00956159/00956159_2.h5',
 '../data/pc_extrusion_labels/0095/00956159/00956159_0.h5',
 '../data/pc_extrusion_labels/0095/00956159/00956159_1.h5')

In [246]:
2397 + 4236 + 2854 + 513

10000

In [215]:
error

['../data/pc_from_vec_labels/0095/00956159/00956159_4.h5',
 '../data/pc_from_vec_labels/0095/00956152/00956152_4.h5',
 '../data/pc_from_vec_labels/0061/00614047/00614047_5.h5',
 '../data/pc_from_vec_labels/0061/00611623/00611623_1.h5',
 '../data/pc_from_vec_labels/0061/00611623/00611623_2.h5',
 '../data/pc_from_vec_labels/0061/00611623/00611623_3.h5',
 '../data/pc_from_vec_labels/0061/00611623/00611623_4.h5',
 '../data/pc_from_vec_labels/0061/00611623/00611623_5.h5',
 '../data/pc_from_vec_labels/0061/00612499/00612499_3.h5',
 '../data/pc_from_vec_labels/0061/00615341/00615341_7.h5',
 '../data/pc_from_vec_labels/0061/00614722/00614722_3.h5',
 '../data/pc_from_vec_labels/0061/00617162/00617162_4.h5',
 '../data/pc_from_vec_labels/0061/00613316/00613316_1.h5',
 '../data/pc_from_vec_labels/0061/00613312/00613312_1.h5',
 '../data/pc_from_vec_labels/0061/00610558/00610558_3.h5',
 '../data/pc_from_vec_labels/0061/00610558/00610558_4.h5',
 '../data/pc_from_vec_labels/0061/00615967/00615967_6.h5

In [281]:
counter = 0
with h5py.File('../data/pc_from_vec_labels/0095/00956159.h5', "r") as f:
    # iterate exactly what's in the file:
    for key in f['sequences'].keys():
        teil = os.path.splitext(path)
        id = path.split("/")[-1].split(".")[0]
        new_path = teil[0] + "/" + id + "_" + key + teil[1]
        all_paths.append(new_path)
        counter += 1

In [282]:
counter

5

In [374]:
with h5py.File("../data/pc_from_vec_labels/0087/00876358.h5") as f:
    print(f.keys())
    labels = f['labels'][:]
    print(np.unique(labels))
    for k,v in f['sequences'].items():
        seq = v[:]
        print(seq)
        print(k, v.shape)

<KeysViewHDF5 ['labels', 'sequences']>
[0 1 2]
[[  4  -1  -1 ...  -1  -1  -1]
 [  0 223 128 ...  -1  -1  -1]
 [  0 223 171 ...  -1  -1  -1]
 ...
 [  3  -1  -1 ...  -1  -1  -1]
 [  3  -1  -1 ...  -1  -1  -1]
 [  3  -1  -1 ...  -1  -1  -1]]
0 (60, 17)
[[  4  -1  -1 ...  -1  -1  -1]
 [  0 223 128 ...  -1  -1  -1]
 [  0 223 150 ...  -1  -1  -1]
 ...
 [  3  -1  -1 ...  -1  -1  -1]
 [  3  -1  -1 ...  -1  -1  -1]
 [  3  -1  -1 ...  -1  -1  -1]]
1 (60, 17)
[[  4  -1  -1 ...  -1  -1  -1]
 [  0 223 128 ...  -1  -1  -1]
 [  0 223 147 ...  -1  -1  -1]
 ...
 [  3  -1  -1 ...  -1  -1  -1]
 [  3  -1  -1 ...  -1  -1  -1]
 [  3  -1  -1 ...  -1  -1  -1]]
2 (60, 17)
[[  4  -1  -1 ...  -1  -1  -1]
 [  0 189 128 ...  -1  -1  -1]
 [  0 189 223 ...  -1  -1  -1]
 ...
 [  3  -1  -1 ...  -1  -1  -1]
 [  3  -1  -1 ...  -1  -1  -1]
 [  3  -1  -1 ...  -1  -1  -1]]
3 (60, 17)


In [247]:
def split_pc_by_labels(pc: np.ndarray, labels: np.ndarray):
    class_pcs = []
    for class_id in np.unique(labels):
        class_mask = (labels == class_id)
        class_pc = pc[class_mask]
        class_pcs.append(class_pc)
    return class_pcs

In [259]:
def find_index(file_id, ds):
    """To find specific files.
    If provided with a file id (str, no extension), returns the index in the dataset.
    """
    pcs = ds.pc
    
    for i, data in enumerate(pcs):
        id = os.path.splitext(os.path.basename(data))[0]#.split("_")[0]

        
        if id == file_id:
            print(i)
            return i
            break

In [257]:
find_index("00956159", a)

173442


173442

In [258]:
train_dataset = PCExtrusionSegmentationDataset("../data", 'train', use_normals=False, verbose=False)

In [358]:
find_index("00876358", train_dataset)

82838


82838

In [274]:
lol = train_dataset[15404]

In [275]:
lol.keys()

dict_keys(['pc', 'label', 'id'])

In [276]:
pc = lol['pc']
label = lol['label']
pc.shape, label.shape, np.unique(label)

(torch.Size([10000, 3]), torch.Size([10000]), array([0, 2, 3, 4, 5]))

In [277]:
aha = split_pc_by_labels(pc, label)

In [278]:
for pc in aha:
    print(pc.shape)

torch.Size([14, 3])
torch.Size([2802, 3])
torch.Size([3134, 3])
torch.Size([3073, 3])
torch.Size([977, 3])


In [283]:
train_dataset = PCExtrusionSegmentationDataset("../data", 'train', use_normals=False, verbose=False)
val_dataset = PCExtrusionSegmentationDataset("../data", 'validation', use_normals=False, verbose=False)
test_dataset = PCExtrusionSegmentationDataset("../data", 'test', use_normals=False, verbose=False)
datasets = [train_dataset, val_dataset, test_dataset]

DATA_DIR = "../data"
error_dict = {}


107/160824

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x10794fd90>>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/myenv/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


226/160824

KeyboardInterrupt: 

In [ ]:

try:




    
    for j, pc in enumerate(pcs):
        pc_path = os.path.join(os.path.abspath(DATA_DIR), "pc_extrusion", id[:4], id, id + "_" + str(j) + ".ply")
        os.makedirs(os.path.dirname(pc_path), exist_ok=True)
        save_pc(pc, pc_path)
        
        h5_path = os.path.join(os.path.abspath(DATA_DIR), "pc_extrusion_labels", id[:4], id, id + "_" + str(j) + ".h5")
        os.makedirs(os.path.dirname(h5_path), exist_ok=True)
        sequence = sequences[str(j)]
        save_h5(j, sequence, h5_path)
        
except Exception as e:
    error_dict[i] = e

In [285]:
data = train_dataset[85223]
id = data['id']
pc = data['pc']
label = data['label']
assert pc.shape[0] == 10000, "Point cloud from pc_from_vec has 10000 points, but got {}".format(pc.shape[0])

In [288]:
id, pc.shape, np.unique(label)

('00956159', torch.Size([10000, 3]), array([0, 1, 2, 4]))

In [293]:
visualize_labeled_pc(pc, label)

[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display
[Open3D WARNING] GLFW Error: Cocoa: Failed to find service port for display


In [289]:
pcs = split_pc_by_labels(pc, label)

sequences = {}
h5_path = os.path.join(DATA_DIR, "pc_from_vec_labels", id[:4], id + ".h5") 

In [290]:
len(pcs)

4

In [292]:
with h5py.File(h5_path, 'r') as f:
    for class_id, sequence in f['sequences'].items():
        print(class_id)
        sequence = sequence[:]
        seq_length = np.where(sequence[:, 0] == 3)[0][0] + 1 # only save up to including the first EOS command
        sequence = sequence[:seq_length, :]
        sequences[class_id] = sequence


0
1
2
3
4


In [417]:
def get_labled_pc_per_ext(sequence, nr_points=8096):
    """ Input:  sequence: Sequence as np (60,17)
                nr_points: Number of points which are sampled per extrusion as int
        Output: merged_pc:            Point cloud where each point is labled by the extrusion that created it as np (N, 3)
                merged_labels:        Labels for each point as np (N)
                extrusion_splits_seq: Extrusion sequence per label as list of np (60,17)
    """
    
    extrusion_splits_seq = split_and_pad_sequence_by_extrusion(sequence)
    point_clouds = []
    for i, ext_seq in enumerate(extrusion_splits_seq):
        pc = seq2pc(ext_seq, nr_points=nr_points)
        print(pc.shape)
        point_clouds.append(pc)
    merged_pc = np.concatenate(point_clouds, axis=0)
        
    labels = []
    for i, pc in enumerate(point_clouds):
        labels.append(np.full((pc.shape[0],), i, dtype=int))
    merged_labels = np.concatenate(labels, axis=0)
    
    return merged_pc, merged_labels, extrusion_splits_seq

In [418]:
def split_and_pad_sequence_by_extrusion(matrix, delimiter=5):
    """Takes (60,17) sequence and splits it by the extrusions and pads it and returns a (60,17) for each extrusion""" 
    matrix = np.array(matrix)
    assert matrix.shape == (60, 17), "Input must be (60, 17)"

    commands = matrix[:,0]
    split_indices = []
    start_idx = 0

    for idx, val in enumerate(commands):
        if val == delimiter:
            split_indices.append((start_idx, idx))
            start_idx = idx + 1

    output = []
    for start, end in split_indices:
        length = 59 - (end - start)
        pad_row = [[3] + 16 * [-1]] * length
        new_matrix = matrix[start:end+1]
       
        pad_matrix = np.vstack([new_matrix, pad_row])
        output.append(pad_matrix)
    return output

In [419]:
abc = PointCloudEmbeddingSequenceDataset("../data", "train")

In [420]:
from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.cadlib.visualize import vec2CADsolid
from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh
from models.DeepCAD.cadlib.extrude import CADSequence
from models.DeepCAD.cadlib.visualize import create_CAD
from models.DeepCAD.cadlib.visualize import CADsolid2pc
from models.DeepCAD.utils.pc_utils import write_ply
import open3d as o3d
from scipy.spatial import cKDTree

In [421]:
def find_index_old(file_id, ds):
    """To find specific files.
    If provided with a file id (str, no extension), returns the index in the dataset.
    """
    pcs = ds.pc
    
    for i, data in enumerate(pcs):
        id = os.path.splitext(os.path.basename(data))[0]
        if id == file_id:
            print(i)
            return i
            break

In [422]:
def seq2pc(sequence, nr_points=8096):
    """ Input:  sequence:  Sequence as np (60,17)
                nr_points: Number of points to sample for the sequence as int
        Output: out_pc:    Output point cloud as np (N, 3)
    """
    
    shape = seq2shape(sequence)
    out_pc = CADsolid2pc(shape, n_points=nr_points)
    return out_pc

In [314]:
find_index_old("00956159", abc)

85309


85309

In [364]:
def seq2shape(seq):
    cad_seq = CADSequence.from_vector(seq, is_numerical=True)
    shape = create_CAD(cad_seq)
    return shape

In [423]:
seq_test = abc[82838]['tgt_vec']
pc = abc[82838]['pc']

In [424]:
merged_pc, merged_labels, ext_splits = get_labled_pc_per_ext(seq_test)

(8096, 3)
(8096, 3)
(8096, 3)
(8096, 3)


In [425]:
np.unique(merged_labels)

array([0, 1, 2, 3])

In [368]:
def filter_pc_and_labels(gt_pc, ext_pc, ext_labels, epsilon=0.01):
    """
    Input: gt_pc: Point cloud sampled from the original shape as np (N, 3)
           ext_pc: Point cloud where points are sampled from all extrusions as np (M, 3)
           ext_labels: Extrusion label for each point in ext_pc
    Output: filtered_pc: ext_pc where only points are retained that are within a distance of epsilon to a point in gt_pc
            filtered_labels: labels for each point in filtered_pc
    """
    filtered_pc, mask = filter_by_nearest_neighbor(gt_pc, ext_pc, epsilon=epsilon)
    filtered_labels = ext_labels[mask]
    return filtered_pc, filtered_labels

In [369]:
def filter_by_nearest_neighbor(pc_A, pc_B, epsilon=0.5):
    """
    Filters point cloud B using nearest neighbors from point cloud A.
    Keeps only points in B that are within `epsilon` distance to any point in A.
    
    :param pc_A: Nx3 point cloud (gt_pc)
    :param pc_B: Mx3 point cloud (pc with labled extrusions)
    
    :return: pc_B filtered point cloud, mask (M,), where mask[i] = True if point i is kept
    """
    tree = cKDTree(pc_A)
    distances, _ = tree.query(pc_B, k=1)
    mask = distances < epsilon
    return pc_B[mask], mask

In [370]:
fpc, flabel = filter_pc_and_labels(pc, merged_pc, merged_labels, epsilon = 0.005)

In [371]:
np.unique(flabel)

array([0, 1, 2, 3])

In [372]:
fpc.shape

(11541, 3)

In [357]:
for i in tqdm(range(1000)):
    seq_test = abc[85309]['tgt_vec']
    pc = abc[85309]['pc']
    merged_pc, merged_labels, ext_splits = get_labled_pc_per_ext(seq_test)
    fpc, flabel = filter_pc_and_labels(pc, merged_pc, merged_labels, epsilon = 0.005)
    if not 3 in flabel:
        print("JOO")
    

100%|███████████████████████████████████████| 1000/1000 [02:14<00:00,  7.45it/s]


In [414]:
import pickle as pkl

with open("../code/error_dict.pkl", "rb") as f:
    a = pkl.load(f)


In [415]:
len(a)

159